In [2]:
import json
import numpy as np
import pandas as pd

In [3]:
TRAIN_DATA_PATH = "db/goemotions/train.tsv"
VALID_DATA_PATH = "db/goemotions/dev.tsv"
TEST_DATA_PATH = "db/goemotions/test.tsv"
EMOTIONS_PATH = "db/goemotions/emotions.txt"
EKMAN_PATH = "db/goemotions/ekman_mapping.json"

TEXT_COLUMN = "text"
LABEL_COLUMN = "emotion"

RANDOM_SEED = 42


In [4]:
train_df = pd.read_csv(TRAIN_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
train_df = train_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("Number of examples:", len(train_df))
print("\nClass distribution:")
print(train_df[LABEL_COLUMN].value_counts())


Number of examples: 43410

Class distribution:
emotion
27           12823
0             2710
4             1873
15            1857
1             1652
             ...  
6,15,22          1
9,10,19          1
7,10,25          1
7,9,24,25        1
0,1,18           1
Name: count, Length: 711, dtype: int64


In [5]:
test_df = pd.read_csv(TEST_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
test_df = test_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("Number of examples:", len(test_df))
print("\nClass distribution:")
print(test_df[LABEL_COLUMN].value_counts())


Number of examples: 5427

Class distribution:
emotion
27         1606
0           348
15          260
4           236
10          195
           ... 
1,8           1
6,12          1
1,15,17       1
4,22,26       1
10,12         1
Name: count, Length: 275, dtype: int64


In [6]:
valid_df = pd.read_csv(VALID_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
valid_df = valid_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("Number of examples:", len(valid_df))
print("\nClass distribution:")
print(valid_df[LABEL_COLUMN].value_counts())


Number of examples: 5426

Class distribution:
emotion
27         1592
0           326
15          261
4           258
10          212
           ... 
2,9,22        1
3,26          1
5,11          1
1,3,18        1
0,17,22       1
Name: count, Length: 296, dtype: int64


In [7]:
with open(EMOTIONS_PATH, encoding="utf-8") as emotions_file:
    labels = [line.strip() for line in emotions_file if line.strip()]

with open(EKMAN_PATH, encoding="utf-8") as ekman_file:
    ekman_mapping = json.load(ekman_file)

    for i in ekman_mapping:
        ekman_mapping[i] = [labels.index(label) for label in ekman_mapping[i]]

print(ekman_mapping)

label2id = {label: index for index, label in enumerate(labels)}
id2label = {index: label for label, index in label2id.items()}
NUM_LABELS = len(labels)

def encode_emotions(emotion_indices):
    encoded = np.zeros(NUM_LABELS, dtype=np.float32)

    for index in str(emotion_indices).split(","):
        encoded[int(index)] = 1.0

    return encoded

def encode_category(emotion_indices):
    encoded = np.zeros(len(ekman_mapping), dtype=np.float32)
    for cat, elems in ekman_mapping.items():
        if any(int(index) in elems for index in emotion_indices):
            encoded[list(ekman_mapping.keys()).index(cat)] = 1.0

    if encoded.sum() > 1:
        print(f"Warning: Multiple categories found for emotion indices {emotion_indices}.")
    return encoded

for df in [train_df, valid_df, test_df]:
    df["label_id"] = df[LABEL_COLUMN].apply(encode_emotions)
    df["category"] = df["label_id"].apply(encode_category)

NUM_LABELS = len(labels)

print("\nLabels:")
print(label2id)


{'anger': [2, 3, 10], 'disgust': [11], 'fear': [14, 19], 'joy': [17, 1, 4, 13, 15, 18, 20, 23, 21, 0, 8, 5], 'sadness': [25, 9, 12, 16, 24], 'surprise': [26, 22, 6, 7]}

Labels:
{'admiration': 0, 'amusement': 1, 'anger': 2, 'annoyance': 3, 'approval': 4, 'caring': 5, 'confusion': 6, 'curiosity': 7, 'desire': 8, 'disappointment': 9, 'disapproval': 10, 'disgust': 11, 'embarrassment': 12, 'excitement': 13, 'fear': 14, 'gratitude': 15, 'grief': 16, 'joy': 17, 'love': 18, 'nervousness': 19, 'optimism': 20, 'pride': 21, 'realization': 22, 'relief': 23, 'remorse': 24, 'sadness': 25, 'surprise': 26, 'neutral': 27}


In [12]:
print("\nClass distribution:")
print(train_df["label_id"])
print(any(train_df["category"].apply(lambda x: sum(x) > 1)))


Class distribution:
0        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
1        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
2        [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
3        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
4        [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
                               ...                        
43405    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
43406    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...
43407    [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
43408    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
43409    [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...
Name: label_id, Length: 43410, dtype: object
False


In [21]:
print(pd.cut(train_df[TEXT_COLUMN].apply(len), bins=[0, 50, 100, 150, 160, 170, 250, 300, 350, 400, 450, 500], right=False).value_counts().sort_index())

text
[0, 50)       15440
[50, 100)     18271
[100, 150)     9223
[150, 160)      305
[160, 170)      117
[170, 250)       50
[250, 300)        0
[300, 350)        0
[350, 400)        0
[400, 450)        1
[450, 500)        0
Name: count, dtype: int64
